# Bibliometria Analítica e Indicadores Científicos
### Atividade prática — `bibliometrix` & Biblioshiny (versão Binder)

---

### 🚀 Você está num ambiente R ao vivo, no navegador

Não precisa instalar nem logar nada — os pacotes já vêm prontos. Funciona como o Colab,
mas com **R de verdade**.

- Para rodar uma célula: clique nela e aperte **Shift + Enter**.
- Rode as células **em ordem, de cima para baixo**.

> ⚠️ **A sessão é temporária.** Se você fechar a aba ou ficar muito tempo parado, o ambiente
> é encerrado e o trabalho se perde. **Baixe o que produzir** (menu *File → Download*) antes de sair.

> ℹ️ **Biblioshiny:** a interface gráfica (`biblioshiny()`) não roda bem aqui (precisa servir um app
> web). No Binder usamos as **mesmas funções** por baixo. Para a interface bonita, use o Posit Cloud
> ou o R instalado no computador.


In [ ]:
# Confirma que estamos num kernel de R
R.version.string

## 1 · Carregar o pacote

Os pacotes já foram instalados quando o Binder montou o ambiente (via `install.R`).
Aqui é só carregar.


In [ ]:
library(bibliometrix)
cat("bibliometrix", as.character(packageVersion("bibliometrix")), "pronto.\n")

## 2 · Carregar o corpus de exemplo

Usamos um corpus já incluído no pacote, para todos conseguirem rodar sem baixar nada.
No trabalho final, troque pela importação dos seus arquivos (próxima seção).


In [ ]:
data(management, package = "bibliometrixData")
M <- management

dim(M)                        # documentos x campos
range(M$PY, na.rm = TRUE)     # intervalo de anos

### Como importar o SEU corpus no Binder

1. No painel de arquivos (à esquerda), use o botão **Upload** para enviar o arquivo exportado
   (ex.: `scopus.bib`).
2. Ele fica na pasta atual; use o nome do arquivo direto no `convert2df()`.

A célula abaixo está desativada (`if (FALSE)`) para não dar erro agora — ative quando tiver o arquivo.


In [ ]:
if (FALSE) {
  M <- convert2df(
    file     = "scopus.bib",
    dbsource = "scopus",   # "wos", "scopus", "openalex", "dimensions", "lens", "pubmed"
    format   = "bibtex"
  )
}

## 3 · Análise descritiva (a base de tudo)

> **🧪 Sua vez 1:** na tabela `MainInformationDF`, ache o **Annual Growth Rate** e a **média de
> citações por documento**.


In [ ]:
results <- biblioAnalysis(M, sep = ";")
S <- summary(results, k = 10, pause = FALSE, verbose = FALSE)
S$MainInformationDF

## 4 · Produção científica anual (evolução temporal)

> **🧪 Sua vez 2:** qual o ano de maior produção? O campo cresce, amadurece ou recua?


In [ ]:
plot(results, k = 10, pause = FALSE)

In [ ]:
S$AnnualProduction

## 5 · Fontes e Lei de Bradford

> **🧪 Sua vez 3:** quantos periódicos formam a zona **Core** (núcleo)?


In [ ]:
BR <- bradford(M)
head(BR$table, 10)

## 6 · Autores e Lei de Lotka

> **🧪 Sua vez 4:** o `Beta` está perto de 2 (padrão clássico de concentração)?


In [ ]:
# A assinatura de lotka() mudou entre versões: versão atual usa lotka(M); antiga, lotka(results).
invisible(capture.output(L <- lotka(M)))
if (!is.list(L)) L <- lotka(results)

L$AuthorProd
cat("\nBeta:", round(L$Beta, 3),
    "| R2:", round(L$R2, 3),
    "| p-valor (KS):", round(L$p.value, 4), "\n")

### Produção dos principais autores ao longo do tempo

> **🧪 Sua vez 5:** quem tem a trajetória mais longa? Quem concentra a produção em poucos anos?


In [ ]:
topAU <- authorProdOverTime(M, k = 10, graph = TRUE)
head(topAU$dfAU)

## 7 · Impacto: índices H, G e M

- **h**: equilíbrio entre volume e impacto · **g**: peso aos muito citados · **m**: corrige pelo tempo.

> **🧪 Sua vez 6:** o autor mais produtivo é também o de maior h-index?


In [ ]:
autores <- gsub(",", " ", names(results$Authors)[1:10])
H <- Hindex(M, field = "author", elements = autores, sep = ";", years = Inf)
H$H[order(-H$H$h_index), ]

## 8 · Documentos e referências mais citados

> **🧪 Sua vez 7:** identifique 2 "clássicos" entre as referências mais citadas.


In [ ]:
CIT <- citations(M, field = "article", sep = ";")
head(CIT$Cited, 10)

## 9 · Tabela-síntese de indicadores

Consolida os indicadores numa só tabela — o coração do tema.


In [ ]:
agr <- S$MainInformationDF$V2[S$MainInformationDF$V1 == "Annual Growth Rate %"][1]

sintese <- data.frame(
  Indicador = c("Documentos (total)", "Autores", "Periodo coberto",
                "Crescimento anual (%)", "Media de citacoes por documento",
                "Documentos por autor"),
  Valor = c(results$Articles,
            length(results$Authors),
            paste(range(M$PY, na.rm = TRUE), collapse = " - "),
            agr,
            round(mean(as.numeric(M$TC), na.rm = TRUE), 2),
            round(results$Articles / length(results$Authors), 2))
)
sintese

# Salva um arquivo (lembre de baixar antes de sair: File > Download)
write.csv(sintese, "indicadores.csv", row.names = FALSE)
cat("Arquivo 'indicadores.csv' salvo na pasta atual.\n")

## 10 · Antes de sair

A sessão do Binder é **temporária**. Para guardar seu trabalho:

- **File → Download** no notebook (e nos arquivos que você gerou, como `indicadores.csv`).

### Desafios finais (discussão)
1. O campo segue a Lei de Lotka (Beta ≈ 2)?
2. O autor mais produtivo é o mais influente?
3. A produção anual indica ascensão, maturidade ou declínio?
4. Qual indicador você defenderia como o **mais importante** para avaliar um corpus?

*Aria, M. & Cuccurullo, C. (2017). bibliometrix. Journal of Informetrics, 11(4), 959-975.*
